In [0]:
%sql
describe workspace.pj_sales.tb_sales_silver

In [0]:
%sql
create table if not exists 
  workspace.pj_sales.tb_fact_sale_gold (
     _sk_sale           bigint generated always as identity
    ,invoice_id         string
    ,part_quantity      int
    ,part_unit_price    double
    ,part_unit_cost     double
    ,part_total_price   double
    ,part_total_cost    double
    ,part_profit        double
    ,_sk_date           bigint
    ,_sk_client         int
    ,_sk_car            int
    ,_sk_store          int
    ,_sk_seller         int
    ,valid_from         timestamp
    ,valid_to           timestamp
  )

In [0]:
%sql
create or replace view vw_fact_sale_sk as
  select
     tss.invoice_id
    ,tss.part_quantity
    ,tss.part_unit_price
    ,tss.part_unit_cost
    ,tss.part_total_price
    ,tss.part_total_cost
    ,tss.part_profit
    ,coalesce(td._sk_date, -1)     as _sk_date
    ,coalesce(ts._sk_client, -1)   as _sk_client
    ,coalesce(tcar._sk_car, -1)    as _sk_car
    ,coalesce(tst._sk_store, -1)   as _sk_store
    ,coalesce(tc._sk_seller, -1)   as _sk_seller
    ,tss.sale_date
    ,tss.ingestion_timestamp
  from      workspace.pj_sales.tb_sales_silver tss
  left join workspace.pj_sales.tb_dim_date_gold td
    on      tss.sale_date = td.date
  left join workspace.pj_sales.tb_dim_seller_gold ts
    on      tss.seller_id = ts.seller_id
    and     ts.valid_to is null
  left join workspace.pj_sales.tb_dim_car_gold tcar
    on      tss.car_brand = tcar.car_brand
    and     tss.car_model = tcar.car_model
    and     tss.car_part = tcar.car_part
    and     tcar.valid_to is null
  left join workspace.pj_sales.tb_dim_store_gold tst
    on      tss.store_id = tst.store_id
    and     tst.valid_to is null
  left join workspace.pj_sales.tb_dim_client_gold tc
    on      tss.client_id = tc.client_id
    and     tc.valid_to is null

In [0]:
%sql
create or replace temp view vw_fact_sale as(
  with dedup as (
    select
       invoice_id
      ,part_quantity
      ,part_unit_price
      ,part_unit_cost
      ,part_total_price
      ,part_total_cost
      ,part_profit
      ,_sk_date
      ,_sk_client
      ,_sk_car
      ,_sk_store
      ,_sk_seller
      ,row_number() over (
        partition by
           invoice_id
        order by 
           sale_date desc
          ,ingestion_timestamp desc
      )                                                             as order_by_date
    from workspace.pj_sales.vw_fact_sale_sk
  )
  select
     invoice_id
    ,part_quantity
    ,part_unit_price
    ,part_unit_cost
    ,part_total_price
    ,part_total_cost
    ,part_profit
    ,_sk_date
    ,_sk_client
    ,_sk_car
    ,_sk_store
    ,_sk_seller
    ,from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')   as valid_from
    ,null                                                           as valid_to
  from dedup
  where order_by_date = 1
)

In [0]:
%sql
create or replace temp view vw_fact_sale_merge as
select
  vfs.*,
  'u' as action
from vw_fact_sale vfs -- Aqui o registro serve para ser atualizado
union all
select
  vfs.*,
  'i' as action
from vw_fact_sale vfs -- Aqui o registro serve para ser inserido

In [0]:
%sql
select * from vw_fact_sale_merge

In [0]:
%sql
merge into workspace.pj_sales.tb_fact_sale_gold tfs
using vw_dim_seller_merge vfsm
on  tfs.seller_id = vfsm.seller_id
and tfs.valid_to is null
and vfsm.action = 'u'
when matched and vfsm.action = 'u' then
  update set
    tfs.valid_to = vfsm.valid_from
when not matched and vfsm.action = 'i' then
  insert (
     invoice_id
    ,part_quantity
    ,part_unit_price
    ,part_unit_cost
    ,part_total_price
    ,part_total_cost
    ,part_profit
    ,_sk_date
    ,_sk_client
    ,_sk_car
    ,_sk_store
    ,_sk_seller
    ,valid_from
    ,valid_to
  )
  values (
     vfsm.invoice_id
    ,vfsm.part_quantity
    ,vfsm.part_unit_price
    ,vfsm.part_unit_cost
    ,vfsm.part_total_price
    ,vfsm.part_total_cost
    ,vfsm.part_profit
    ,vfsm._sk_date
    ,vfsm._sk_client
    ,vfsm._sk_car
    ,vfsm._sk_store
    ,vfsm._sk_seller
    ,vfsm.valid_from
    ,vfsm.valid_to
  )


In [0]:
%sql
select * from pj_sales.tb_fact_sale_gold